[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/12_linear_attention.ipynb)

# 🔴 Hard: Linear Self-Attention

*Attention & Transformers*
Implement **linear attention**: replace $\text{softmax}(QK^\top)V$ with a
kernel feature map so the sequence length drops out of the complexity.

Standard attention computes
$$\text{softmax}\!\left(\tfrac{QK^\top}{\sqrt{d}}\right)V$$
which materializes a $T \times T$ matrix. Linear attention replaces the softmax
kernel with $\phi(q)^\top\phi(k)$ for a feature map $\phi$, giving

$$O_i = \frac{\phi(q_i)^\top \sum_j \phi(k_j) v_j^\top}
             {\phi(q_i)^\top \sum_j \phi(k_j)}$$

Use $\phi(x) = \text{elu}(x) + 1$, which is strictly positive — required, since
it plays the role of an unnormalised probability.

### Signature
```python
def linear_attention(q, k, v):
    # q: (..., T_q, d), k: (..., T_k, d), v: (..., T_k, d_v)
    ...  # -> (..., T_q, d_v)
```

### Rules
- Never form a `(T_q, T_k)` matrix — that defeats the entire point
- Contract $K$ with $V$ **first**
- Include the denominator; without it this is not an average and the output
  scale drifts with sequence length
- Add a small epsilon to the denominator
- Do not use `jax.nn.softmax`

### The whole trick is associativity
$(QK^\top)V$ and $Q(K^\top V)$ are the same product, but:

| Order | Intermediate | Cost |
|---|---|---|
| $(QK^\top)V$ | $T \times T$ | $O(T^2 d)$ |
| $Q(K^\top V)$ | $d \times d_v$ | $O(T d\, d_v)$ |

Softmax is what forbids the second grouping — it is a nonlinearity *between*
$QK^\top$ and the multiplication by $V$. Remove it, and matrix multiplication
becomes reassociable. Every linear-attention variant (Performer, RFA, and the
linear-attention view of state-space models) is a different choice of $\phi$
around that one observation.

### What you give up
The $d \times d_v$ summary is a **fixed-size** state, no matter how long the
sequence. So this is lossy in a way softmax attention is not: softmax can
sharply retrieve a single token out of a million (its "state" grows with $T$),
while linear attention compresses everything into the same matrix and cannot.
That is precisely why linear-attention models underperform on retrieval-heavy
tasks like needle-in-a-haystack, and why the strongest recent designs interleave
a few full-attention layers among the linear ones.

The flip side is the reason to care: because the state is fixed-size and updates
additively, the **causal** version runs as an RNN at inference — $O(1)$ memory
per token instead of a KV cache that grows without bound ([[kv_cache]]).

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


def linear_attention(q, k, v):
    """Linear attention with the phi(x) = elu(x) + 1 feature map.

    Args:
        q: (..., T_q, d)
        k: (..., T_k, d)
        v: (..., T_k, d_v)

    Returns:
        (..., T_q, d_v)
    """
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

q = jax.random.normal(jax.random.key(0), (2, 16, 8))
k = jax.random.normal(jax.random.key(1), (2, 16, 8))
v = jax.random.normal(jax.random.key(2), (2, 16, 4))

out = linear_attention(q, k, v)
print("q:", q.shape, " v:", v.shape, " -> out:", out.shape)

# Cost grows linearly, not quadratically, in T.
for T in (64, 256, 1024):
    qq = jax.random.normal(jax.random.key(3), (1, T, 8))
    vv = jax.random.normal(jax.random.key(4), (1, T, 4))
    print(f"  T={T:>5}: out {linear_attention(qq, qq, vv).shape}")

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution, status

check("linear_attention")

# hint("linear_attention")      # stuck? nudge without the answer
# solution("linear_attention")  # spoiler: the reference implementation
# status()                      # your dashboard across all problems